# COE (Chain of Errors) Prediction
Test individual functions and full pipeline from `revlm.metrics.utils.e_gen`.

In [1]:
import os
import sys
from argparse import Namespace
import torch

repo_root = "/scratch/jq2uw/MME/instruct_vlm_edit"
os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.append(repo_root)

from revlm import *
from revlm.run.edit_utils import find_errors
from revlm.metrics.utils.e_gen import (
    parse_cot_sentences,
    verify_sentence,
    process_sample,
    coe_prediction,
    print_coe_results,
)

In [2]:
# Config - adjust model_name, dataset_name as needed
args = Namespace(
    config="revlm/config/config.yaml",
    editor="baseline",
    model_name="qwen3",
    dataset_name="fvqa",
    task="mc",
    split="all",
    n_iter=1,
    ckpt_dir=None,
    task_dir=None,
    edit_dir=None,
    pred_dir=None,
    pred_postedit_dir=None,
    suffix="",
    subsample=0,
    dropout=None,
    pred_by="label_maxprob",
    device=None,
)

config = configure_args(args, config_path=args.config)
config.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
config.subsample = args.subsample
config.cot = True
config.rationale = False
config.overwrite = False

print(f"Model: {config.model.name}")
print(f"Dataset: {config.experiment.dataset_name}")

Task evaluation metrics will be saved to results/te/baseline/Qwen3-VL-8B-Instruct/fvqa
Edit evaluation metrics will be saved to results/ee/baseline/Qwen3-VL-8B-Instruct/fvqa
Predictions will be saved to results/pred/Qwen3-VL-8B-Instruct/fvqa
Post-edit predictions will be saved to results/pred_postedit/baseline/Qwen3-VL-8B-Instruct/fvqa
Unified filename to save: mc_all.json
Model: Qwen/Qwen3-VL-8B-Instruct
Dataset: fvqa


In [3]:
# Load model and error samples
model, edit_ds = find_errors(config)
model.model.eval()

errors = edit_ds.data
print(f"Model: {model.device}")
print(f"Error samples: {len(errors)}")

Step 1 (predictions)
Total samples 5826 loaded from results/pred/Qwen3-VL-8B-Instruct/fvqa/mc_all.json
getting edits predicted by: label_maxprob

model_old predictions:
{'uid': '11', 'image': 'data/images/fvqa/COCO_val2014_000000106920.jpg', 'question': 'What can you find on the pizzas in this this image?', 'answer': 'cheese', 'rationale': 'Something you find on a pizza is cheese.', 'cot': 'The image shows pizzas. The pizzas have a topping that is light in color and melted. Cheese is a common topping on pizzas that is light in color and melts when cooked.', 'choices': 'chocolate; pepperoni; cheese; pickles', 'idx_choices': '(A) chocolate\n(B) pepperoni\n(C) cheese\n(D) pickles', 'idx': 0, 'gold': {'label': 'cheese', 'choices': {'str': 'chocolate; pepperoni; cheese; pickles', 'ls': ['chocolate', 'pepperoni', 'cheese', 'pickles']}, 'label_train': 'cheese'}, 'prompt': 'What can you find on the pizzas in this this image?', 'pred': {'answer': "Based on the image provided, here's what you", 

## Test Individual Functions

In [4]:
# Test parse_cot_sentences
ex = errors[0]
cot = ex.get('cot', '')
sentences = parse_cot_sentences(cot)

print(f"COT: {cot}")
print(f"Sentences ({len(sentences)}):")
for i, s in enumerate(sentences):
    print(f"  [{i}] {s}")

COT: The image shows pizzas. The pizzas have a topping that is light in color and melted. Cheese is a common topping on pizzas that is light in color and melts when cooked.
Sentences (3):
  [0] The image shows pizzas.
  [1] The pizzas have a topping that is light in color and melted.
  [2] Cheese is a common topping on pizzas that is light in color and melts when cooked.


In [5]:
# Test verify_sentence on first sentence
if sentences:
    flag, p_yes, p_no = verify_sentence(model, ex['image'], sentences[0])
    print(f"Sentence: {sentences[0]}")
    print(f"P(yes)={p_yes:.3f}, P(no)={p_no:.3f}, error={flag}")

Sentence: The image shows pizzas.
P(yes)=1.000, P(no)=0.000, error=0


In [7]:
# Test process_sample
ex_copy = ex.copy()
ex_copy = process_sample(model, ex_copy)

coe = ex_copy['coe_pred']
print(f"uid: {ex_copy['uid']}")
print(f"Question: {ex_copy['question']}")
print(f"Gold: {ex_copy['gold']['label']}, Pred: {ex_copy['pred']['label_maxprob']}")
print(f"Sentences: {coe['sentences']}")
print(f"Subsets ({len(coe['subsets'])}):")
for sub in coe['subsets']:
    mark = 'x' if sub['error'] else 'v'
    print(f"  [{mark}] {sub['indices']} p_yes={sub['p_yes']:.2f} p_no={sub['p_no']:.2f}")

uid: 11
Question: What can you find on the pizzas in this this image?
Gold: cheese, Pred: pepperoni
Sentences: ['The image shows pizzas.', 'The pizzas have a topping that is light in color and melted.', 'Cheese is a common topping on pizzas that is light in color and melts when cooked.']
Subsets (7):
  [v] [0] p_yes=1.00 p_no=0.00
  [x] [1] p_yes=0.01 p_no=0.99
  [v] [2] p_yes=1.00 p_no=0.00
  [x] [0, 1] p_yes=0.00 p_no=1.00
  [v] [0, 2] p_yes=1.00 p_no=0.00
  [v] [1, 2] p_yes=1.00 p_no=0.00
  [v] [0, 1, 2] p_yes=1.00 p_no=0.00


## Run Full Pipeline

In [8]:
# Run COE prediction on all errors and save
results = coe_prediction(model, edit_ds, config)

Processing 857 error samples for COE prediction...


KeyboardInterrupt: 

In [ ]:
# Inspect results
print_coe_results(results, max_print=10)